# TD 3 : prédire la réadmission, et décider quoi en faire

## Comment se passe cette journée

Vous travaillez **en binôme**, et vous changez de clavier à chaque étape. Celui qui ne tape pas lit l'énoncé à voix haute et vérifie.

**Personne ne lève la main pour savoir s'il a juste.** Chaque exercice se termine par une cellule de vérification qui vous le dit, et qui explique ce qui ne va pas si ce n'est pas bon. Les indices sont repliés sous chaque exercice : ouvrez-les plutôt que d'attendre.

**Quand vous finissez une étape**, marquez-la sur le tableau au mur. C'est ce qui permet de voir où en est la salle sans faire le tour des rangs.

## Ce que vous saurez faire

1. repérer une target mal définie avant de modéliser
2. dire pourquoi une accuracy de 88,6 % ne vaut rien ici
3. lire une matrice de confusion en euros et pas en pourcentages
4. calculer un seuil au lieu de le régler
5. produire une recommandation qui résiste à un désaccord sur les coûts

## Prérequis

- avoir fini les TD 1 et 2
- avoir suivi le bloc C de la journée 1

**Durée :** 100 minutes

## Le contexte

Un patient sort de l'hôpital. Reviendra-t-il sous 30 jours ? La réadmission précoce coûte cher et signale souvent une sortie organisée trop vite. 101 766 séjours réels, en open data. Le problème est difficile et personne ne le résout bien, ce qui le rend intéressant.

## Le déroulé

| quand | quoi |
|---|---|
| 3:45 | étape 1 : la target, et le piège qu'elle cache |
| 4:05 | étape 2 : le chiffre qui ment |
| 4:35 | étape 3 : le seuil, et l'hypothèse cachée |
| 5:10 | étape 4 : la recommandation |
| 5:30 | pause |

In [ ]:
# Amorce : a executer en premier, une seule fois par notebook.
import sys, pathlib
racine = pathlib.Path.cwd()
while not (racine / "aeolis").is_dir() and racine != racine.parent:
    racine = racine.parent
sys.path.insert(0, str(racine))
import os
os.chdir(racine)

import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from aeolis import plots, evaluate as ev

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)

# Les donnees sont dans cours_fr/data pendant le cours, et dans donnees/ dans le
# dossier qui vous est remis. On prend celui qui existe, pour que le meme notebook
# fonctionne dans les deux cas.
DONNEES = next((c for c in (racine / "cours_fr" / "data", racine / "donnees",
                            racine / "data") if c.is_dir()), racine)

print("racine du cours :", racine.name)
print("donnees :", DONNEES.name, "->",
      sorted(p.name for p in DONNEES.glob("*.parquet")) or "aucun parquet")
print("pret")

---

## Étape 1 : La target, et le piège qu'elle cache

**20 minutes.** Avant de prédire un événement, se demander qui peut physiquement le vivre.

## Mémo de code

Chaque exercice ci-dessous se résout en adaptant un de ces patrons. Copiez, changez les noms, exécutez. Ce n'est pas tricher : c'est ainsi que l'on écrit du code de data.

```python
# charger et convertir une colonne texte en nombres
sejours['col'] = pd.to_numeric(sejours['col'])

# fabriquer une cible binaire
cible = (sejours['readmitted'] == '<30').astype(int)

# compter les valeurs d'une colonne
sejours['col'].value_counts()

# garder les lignes qui ne sont PAS dans une liste
garde = sejours[~sejours['col'].isin(['11', '19'])]

# split groupe par patient
tr, te = next(GroupShuffleSplit(
    n_splits=1, test_size=0.25, random_state=42)
    .split(X, y, groupes))

# probabilites plutot que verdicts
proba = modele.predict_proba(X_te)[:, 1]

# la matrice de confusion a un seuil choisi
tn, fp, fn, tp = confusion_matrix(y_te, (proba >= seuil).astype(int)).ravel()
```

### 1.1 Qui sort de l'hôpital ?

La colonne `discharge_disposition_id` code le mode de sortie. Regardez ce qu'elle contient avant de modéliser quoi que ce soit.

In [ ]:
# A FAIRE : charger le fichier et compter les modes de sortie.
sejours = ...
modes = ...

print(modes.head(12))

**Indice 1.** Le fichier est `DONNEES / 'readmission.parquet'`, ou DONNEES vient de l'amorce. Puis `value_counts()` sur `discharge_disposition_id`.

**Indice 2.** `sejours = pd.read_parquet(DONNEES / 'readmission.parquet')` puis `modes = sejours['discharge_disposition_id'].value_counts()`

In [ ]:
assert len(sejours) == 101766, f'attendu 101 766 sejours, obtenu {len(sejours)}'
print('Les codes 11, 19, 20 et 21 designent un DECES.')
print('Les codes 13 et 14 designent des soins palliatifs ou un hospice.')
deces = sejours['discharge_disposition_id'].isin(['11', '19', '20', '21']).sum()
pall = sejours['discharge_disposition_id'].isin(['13', '14']).sum()
print()
print(f'sejours termines par un deces           : {deces:,}')
print(f'sejours termines en soins palliatifs    : {pall:,}')
print()
print('Question : un patient decede peut-il etre readmis sous 30 jours ?')

### 1.2 Retirer ce qui ne peut pas arriver

Un patient décédé ne sera jamais réadmis. Le laisser dans le tableau apprend au modèle que « décédé » prédit « pas de réadmission ». C'est vrai, c'est inutile, et c'est dangereux.

In [ ]:
# A FAIRE (1) : garder les sejours qui ne sont NI un deces NI des soins palliatifs.
exclus = ['11', '19', '20', '21', '13', '14']
vivants = ...

# A FAIRE (2) : la cible binaire, 1 si readmission sous 30 jours.
y = ...

print(f'sejours retenus : {len(vivants):,}')
print(f'taux de readmission : {y.mean():.4f}')

**Indice 1.** Pour (1), le patron `~...isin(...)` du mémo. Pour (2), comparer `readmitted` à `'<30'` et convertir en entier.

**Indice 2.** `vivants = sejours[~sejours['discharge_disposition_id'].isin(exclus)]` puis `y = (vivants['readmitted'] == '<30').astype(int).values`

In [ ]:
assert len(vivants) == 99343, (
    f'attendu 99 343 sejours retenus, obtenu {len(vivants):,}')
assert abs(y.mean() - 0.1139) < 0.002, (
    f'attendu un taux de 0,1139, obtenu {y.mean():.4f}. '
    'Avez-vous bien compare a <30 et non a >30 ?')
print(f'correct : {len(vivants):,} sejours, taux {y.mean():.2%}.')
print(f'Avant nettoyage le taux etait de 11,16 %. Apres, {y.mean():.2%}.')
print()
print(f'RETENEZ : répondre toujours non donne {1 - y.mean():.2%} d accuracy.')

---

## Étape 2 : Le chiffre qui ment

**30 minutes.** Quatre modèles, une accuracy identique, et des AUC très différentes.

### 2.1 Construire les features

In [ ]:
# Fourni : les colonnes retenues. Le parquet stocke tout en texte,
# il faut donc convertir les numeriques.
NUM = ['time_in_hospital', 'num_lab_procedures', 'num_procedures',
       'num_medications', 'number_outpatient', 'number_emergency',
       'number_inpatient', 'number_diagnoses']
CAT = ['race', 'gender', 'age', 'admission_type_id', 'admission_source_id',
       'A1Cresult', 'max_glu_serum', 'change', 'diabetesMed',
       'insulin', 'metformin']

# A FAIRE : construire X. Les NUM en nombres, les CAT en category,
#           avec les '?' et les vides remplaces par 'Inconnu'.
X = ...

print(X.shape, '  types :', X.dtypes.value_counts().to_dict())

**Indice 1.** Partez de `vivants[NUM + CAT].copy()`. Pour les NUM, `pd.to_numeric`. Pour les CAT, `.replace('?', 'Inconnu').fillna('Inconnu').astype('category')`.

**Indice 2.** Une boucle `for c in NUM:` puis une boucle `for c in CAT:`, en assignant `X[c] = ...` à chaque fois.

In [ ]:
assert X.shape == (99343, 19), f'attendu (99343, 19), obtenu {X.shape}'
numeriques = sum(1 for c in NUM if X[c].dtype.kind in 'if')
assert numeriques == 8, (
    f'{numeriques} colonnes numeriques sur 8. pd.to_numeric a-t-il ete applique ?')
assert (X['A1Cresult'] == 'Inconnu').sum() > 80000, (
    'A1Cresult est vide a 85 %, ces vides doivent devenir la categorie Inconnu')
print('correct : 8 colonnes numeriques, 11 categorielles.')
print(f"A1Cresult vaut 'Inconnu' {(X['A1Cresult'] == 'Inconnu').sum():,} fois :")
print('le dosage n a pas ete prescrit, et cela dit quelque chose du patient.')

### 2.2 Quatre modèles, et une surprise

**Pariez avant de lancer.** Lequel aura la meilleure accuracy ?

In [ ]:
# A FAIRE : entrainer les quatre et remplir le tableau.
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit

# Un patient peut avoir PLUSIEURS sejours. Un split par ligne le mettrait
# des deux cotes, et le modele reverrait au test des gens appris par coeur.
groupes = vivants['patient_nbr'].values
tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.25,
                                random_state=42).split(X, y, groupes))
X_tr, X_te, y_tr, y_te = X.iloc[tr], X.iloc[te], y[tr], y[te]

bidon = DummyClassifier(strategy='most_frequent').fit(X_tr, y_tr)
arbre = ...

proba_arbre = ...

print(f"{'modele':<22}{'accuracy':>10}{'recall':>9}{'AUC':>8}")
for nom, pred, proba in [
        ('toujours non', bidon.predict(X_te), np.full(len(y_te), 0.0)),
        ('gradient boosting', arbre.predict(X_te), proba_arbre)]:
    print(f'{nom:<22}{accuracy_score(y_te, pred):>10.4f}'
          f'{recall_score(y_te, pred, zero_division=0):>9.4f}'
          f'{roc_auc_score(y_te, proba):>8.4f}')

**Indice 1.** `HistGradientBoostingClassifier(random_state=42, categorical_features=CAT)` accepte directement les colonnes `category`.

**Indice 2.** `arbre = HistGradientBoostingClassifier(random_state=42, categorical_features=CAT).fit(X_tr, y_tr)` puis `proba_arbre = arbre.predict_proba(X_te)[:, 1]`

In [ ]:
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score
acc_bidon = accuracy_score(y_te, bidon.predict(X_te))
acc_arbre = accuracy_score(y_te, arbre.predict(X_te))
auc_arbre = roc_auc_score(y_te, proba_arbre)
rec_arbre = recall_score(y_te, arbre.predict(X_te), zero_division=0)
assert abs(acc_bidon - 0.8862) < 0.005
assert abs(auc_arbre - 0.6345) < 0.02, (
    f'AUC attendue vers 0,63, obtenue {auc_arbre:.4f}')
print(f'ecart d accuracy entre le modele et la baseline : '
      f'{acc_arbre - acc_bidon:+.4f}')
print(f'ecart d AUC                                    : '
      f'{auc_arbre - 0.5:+.4f}')
print()
print(f'Le modele attrape {rec_arbre:.2%} des readmissions au seuil 0,50.')
print('Son accuracy est la meilleure du tableau. Il est pourtant inutile.')

### 2.4 Les quatre cases, et les cinq formules

Le cours a montré la matrice au seuil 0,50. Reconstruisez-la, puis calculez les cinq métriques **à la main**, à partir des quatre nombres seulement. Aucune fonction toute faite : c'est le seul moyen de voir pourquoi elles divergent.

In [ ]:
# A FAIRE : les quatre cases, puis les cinq metriques, a la main.
from sklearn.metrics import confusion_matrix

vn, fp, fn, vp = confusion_matrix(
    y_te, (proba_arbre >= 0.50).astype(int)).ravel()

accuracy    = ...      # (VP + VN) / tout
precision   = ...      # VP / (VP + FP)
recall      = ...      # VP / (VP + FN)
specificity = ...      # VN / (VN + FP)
f1          = ...      # 2 x precision x recall / (precision + recall)

print(f'VP {vp:,}   FP {fp:,}   FN {fn:,}   VN {vn:,}')
for nom, v in [('accuracy', accuracy), ('precision', precision),
               ('recall', recall), ('specificity', specificity),
               ('F1', f1)]:
    print(f'{nom:<12}{v:.4f}')

**Indice 1.** `confusion_matrix(...).ravel()` rend les quatre dans l'ordre vn, fp, fn, vp. Ce n'est PAS l'ordre dans lequel la matrice s'affiche, et c'est le piège.

**Indice 2.** `accuracy = (vp + vn) / (vp + fp + fn + vn)`, `precision = vp / (vp + fp)`, `recall = vp / (vp + fn)`, `specificity = vn / (vn + fp)`, `f1 = 2 * precision * recall / (precision + recall)`

In [ ]:
assert abs(accuracy - 0.8863) < 0.01, f'accuracy {accuracy:.4f}'
assert abs(specificity - 0.9991) < 0.01, f'specificity {specificity:.4f}'
assert recall < 0.05, (
    f'recall attendu tres bas, obtenu {recall:.4f}. '
    'Avez-vous inverse vp et vn dans le ravel() ?')
assert f1 < 0.10, f'F1 {f1:.4f}'
print('correct. Les memes quatre nombres, et deux verdicts opposes :')
print(f'  accuracy {accuracy:.4f} et specificity {specificity:.4f} disent excellent,')
print(f'  recall {recall:.4f} et F1 {f1:.4f} disent catastrophique.')
print()
moyenne = (precision + recall) / 2
print(f'RETENEZ : la moyenne ARITHMETIQUE vaudrait {moyenne:.4f} et masquerait')
print('le probleme. F1 est harmonique, donc elle suit toujours la plus faible.')

### 2.5 La courbe ROC, et la vraie définition de l'AUC

Tracez la courbe, puis **vérifiez** ce que l'AUC veut dire au lieu de le croire. La définition est simple : prenez au hasard un patient réadmis et un patient qui ne l'a pas été, l'AUC est la probabilité que le modèle donne le score le plus élevé au premier. Un tirage suffit à le montrer.

In [ ]:
# A FAIRE (1) : tracer la courbe ROC, avec la diagonale du hasard.
from sklearn.metrics import roc_curve

fpr, tpr, seuils = roc_curve(y_te, proba_arbre)

plt.figure(figsize=(5, 5))
...                        # la courbe
...                        # la diagonale, en pointilles
plt.xlabel('faux positifs')
plt.ylabel('recall')
plt.title(f'AUC {auc_arbre:.4f}, sur {len(seuils):,} seuils')
plt.show()

# A FAIRE (2) : verifier la lecture probabiliste par tirage.
rng = np.random.default_rng(42)
scores_positifs = proba_arbre[y_te == 1]
scores_negatifs = proba_arbre[y_te == 0]
a = rng.choice(scores_positifs, 200_000)
b = rng.choice(scores_negatifs, 200_000)
par_tirage = ...           # la part des paires ou a passe devant b

print(f'par tirage : {par_tirage:.4f}')
print(f'par sklearn : {auc_arbre:.4f}')

**Indice 1.** Pour (1), `plt.plot(fpr, tpr)` puis `plt.plot([0, 1], [0, 1], '--')`. Pour (2), `(a > b)` est un tableau de booleens, et la moyenne d'un tableau de booleens est une proportion.

**Indice 2.** `par_tirage = float((a > b).mean() + 0.5 * (a == b).mean())`. Le demi-point pour les egalites n'est pas un detail : beaucoup de patients partagent exactement le meme score.

In [ ]:
assert abs(par_tirage - auc_arbre) < 0.01, (
    f'les deux devraient coincider : {par_tirage:.4f} contre {auc_arbre:.4f}. '
    'Avez-vous compte le demi-point des egalites ?')
print(f'correct. {par_tirage:.4f} par tirage, {auc_arbre:.4f} par sklearn.')
print()
print('RETENEZ : l AUC n est pas une note abstraite. C est la probabilite')
print('qu un readmis soit mieux classe qu un non readmis, et vous venez')
print(f'de la retrouver sans jamais appeler roc_auc_score.')
print()
print(f'Le hasard donnerait 0,5000. Vous avez {par_tirage:.4f}.')

---

## Étape 3 : Le seuil, et l'hypothèse cachée

**30 minutes.** Deux calculs de coût, deux réponses opposées, et une seule différence entre les deux.

### 3.1 La matrice de coût, premier essai

Hypothèses : une réadmission manquée coûte **8 000 EUR**, une fausse alerte **80 EUR**. Ce sont des hypothèses, pas des mesures.

In [ ]:
# A FAIRE : le cout total pour un seuil donne, puis le seuil le moins cher.
from sklearn.metrics import confusion_matrix

COUT_READMISSION, COUT_APPEL = 8000.0, 80.0

def cout(seuil):
    ...

grille = np.linspace(0.01, 0.99, 99)
couts = [cout(s) for s in grille]
seuil_optimal = grille[int(np.argmin(couts))]

print(f'cout au seuil 0,50 : {cout(0.50):>14,.0f} EUR')
print(f'seuil le moins cher : {seuil_optimal:.2f}')
print(f'cout a ce seuil     : {min(couts):>14,.0f} EUR')

**Indice 1.** Le patron du mémo donne `tn, fp, fn, tp`. Le coût est `fp * COUT_APPEL + fn * COUT_READMISSION`.

**Indice 2.** Dans `cout`, faites `tn, fp, fn, tp = confusion_matrix(y_te, (proba_arbre >= seuil).astype(int)).ravel()` puis renvoyez la somme pondérée.

In [ ]:
assert seuil_optimal < 0.10, (
    f'le seuil optimal devrait etre tres bas, vous trouvez {seuil_optimal:.2f}')
part_alertee = (proba_arbre >= seuil_optimal).mean()
economie = 1 - min(couts) / cout(0.50)
print(f'economie affichee : {economie:.1%}')
print(f'part de patients alertes a ce seuil : {part_alertee:.1%}')
print()
assert part_alertee > 0.95, 'le seuil optimal devrait alerter presque tout le monde'
print('Le seuil optimal dit d appeler TOUT LE MONDE.')
print('Dans ce cas le modele ne sert plus a rien du tout.')
print('Quand l optimum consiste a tout declencher, ce n est pas le seuil')
print('qui est bon : c est l hypothese qui est fausse.')

### 3.2 L'hypothèse cachée, et la version honnête

La matrice à deux nombres suppose **qu'un appel évite toujours la réadmission**. C'est faux, et c'est pour cela qu'elle conclut à l'absurde.

Version honnête : un programme de sortie coûte **500 EUR** par patient inscrit et évite **20 %** des réadmissions de ceux qu'il inscrit.

In [ ]:
# A FAIRE (1) : au-dessus de quel risque inscrire un patient est-il rentable ?
COUT_PROGRAMME, EFFICACITE = 500.0, 0.20
seuil_rentable = ...

# A FAIRE (2) : le benefice net du programme a un seuil donne.
def benefice(seuil):
    ...

benefices = [benefice(s) for s in grille]
meilleur_seuil = grille[int(np.argmax(benefices))]

print(f'rentable au-dessus de  : {seuil_rentable:.1%} de risque')
print(f'meilleur seuil         : {meilleur_seuil:.2f}')
print(f'benefice net           : {max(benefices):>12,.0f} EUR')
print(f'inscrire tout le monde : {benefice(0.0):>12,.0f} EUR')

**Indice 1.** Pour (1) : inscrire est rentable quand le gain attendu dépasse le coût, donc quand `EFFICACITE * risque * COUT_READMISSION > COUT_PROGRAMME`. Isolez `risque`.

**Indice 2.** `seuil_rentable = COUT_PROGRAMME / (EFFICACITE * COUT_READMISSION)`. Pour (2), les inscrits sont `proba_arbre >= seuil`, les réadmissions évitées `EFFICACITE * y_te[inscrits].sum()`.

In [ ]:
assert abs(seuil_rentable - 0.3125) < 0.001, (
    f'attendu 31,25 %, obtenu {seuil_rentable:.4f}. '
    'Le calcul est COUT_PROGRAMME / (EFFICACITE * COUT_READMISSION).')
assert benefice(0.0) < 0, 'inscrire tout le monde doit couter de l argent'
assert max(benefices) > 0, 'il doit exister un seuil rentable'
inscrits = int((proba_arbre >= meilleur_seuil).sum())
print(f'correct. {inscrits} patients inscrits sur {len(y_te):,}, '
      f'soit {inscrits / len(y_te):.1%}.')
print()
print('La matrice naive disait : appeler tout le monde, economiser 92 %.')
print(f'Le calcul honnete dit : inscrire tout le monde coute '
      f'{benefice(0.0):,.0f} EUR.')
print('Meme modele, memes donnees, meme jour. Une hypothese explicitee.')

---

## Étape 4 : La recommandation qui résiste

**20 minutes.** Un livrable qui tient même si votre interlocuteur conteste vos hypothèses.

### 4.1 Le rendement par budget d'appels

Une infirmière peut faire un nombre limité d'appels. Cette contrainte-là ne se discute pas, contrairement au prix d'une réadmission.

In [ ]:
# A FAIRE : pour 200, 500 et 1000 patients les mieux classes,
#           combien de readmissions trouve-t-on ?
for budget in (200, 500, 1000):
    ordre = ...
    trouvees = ...
    print(f'les {budget:>4} mieux classes : {trouvees:>4} readmissions '
          f'({trouvees / budget:>6.1%} de rendement)')

print(f'\npour memoire, au hasard : {y_te.mean():.1%}')

**Indice 1.** `np.argsort(-proba_arbre)[:budget]` donne les indices des mieux classés. Puis sommez `y_te` sur ces indices.

**Indice 2.** `ordre = np.argsort(-proba_arbre)[:budget]` et `trouvees = int(y_te[ordre].sum())`

In [ ]:
ordre = np.argsort(-proba_arbre)[:200]
rendement = int(y_te[ordre].sum()) / 200
assert rendement > 0.35, (
    f'rendement de {rendement:.1%} sur le top 200, attendu nettement plus')
print(f'Les 200 mieux classes contiennent {rendement:.1%} de readmissions,')
print(f'contre {y_te.mean():.1%} au hasard, soit un facteur '
      f'{rendement / y_te.mean():.1f}.')
print()
print('Ce resultat ne depend d AUCUNE hypothese de prix.')
print('C est celui que vous pouvez defendre devant quelqu un qui conteste')
print('vos chiffres de cout, et c est donc celui qu il faut mettre en avant.')

### 4.2 Écrivez la recommandation  *(niveau 3)*

La responsable du service vous demande quoi faire. Elle ne sait pas ce qu'est une AUC et elle n'a pas à le savoir.

**Six ou sept phrases**, dans la cellule ci-dessous. Pas de squelette, pas de bonne réponse unique : la note porte sur le fait qu'une personne qui ne connaît pas le sujet puisse agir.

In [ ]:
# A FAIRE : la recommandation. Six ou sept phrases, a la responsable du service.
#           Utilisez les nombres que vous avez mesures, pas des adjectifs.
recommandation = '...'

**Indice 1.** Partez de son côté du bureau. Elle ne veut pas un modèle, elle veut savoir quels patients suivre lundi, combien, et si cela vaut mieux que ce qu'elle fait déjà.

**Indice 2.** Les trois nombres qui lui parlent sont la taille de la liste, le rendement sur cette liste, et le rendement sans la liste. L'AUC n'en fait pas partie.

In [ ]:
from aeolis.checks import done
done(recommandation)
assert len(recommandation) > 400, 'six ou sept phrases, pas un titre'
print('Comparez avec la solution. La note porte sur cinq choses :')
print('  - une action concrete : combien de patients, a quelle frequence')
print('  - le rendement ET le point de comparaison, dans SES unites')
print('  - la limite annoncee avant qu elle ne la decouvre')
print('  - ce que vous lui demandez, et pourquoi le releve est l actif')
print('  - la precaution d echantillonnage : une liste seule biaise le releve')

## Ce que cette étape a établi

- Un patient décédé ne peut pas être réadmis : 2 423 séjours retirés, et c'est une faute de target, pas de code.
- Taux de réadmission 11,39 %, donc 88,61 % d'accuracy sans rien apprendre.
- L'accuracy ne distingue pas les modèles ici. L'AUC, si : 0,50 contre 0,64.
- Une matrice de coût à deux nombres cache une hypothèse d'efficacité, et elle conclut à appeler tout le monde.
- Le livrable robuste est le rendement par budget : 42,5 % sur 200 patients contre 11,4 % au hasard, sans aucune hypothèse de prix.

## Les erreurs classiques

- **Annoncer une accuracy sans son taux de base.** Ici 88,6 % est le plancher, pas un résultat.
- **Régler le seuil à l'oeil.** Il se calcule : coût du programme divisé par efficacité fois coût évité.
- **Accepter une matrice de coût sans chercher l'hypothèse cachée.** Elle suppose toujours que l'intervention fonctionne, et elle décidera à votre place.

## Pour aller plus loin, si vous avez fini en avance

- Refaire l'étape 3 avec une efficacité de 10 % puis de 40 %. À partir de quelle efficacité le programme cesse-t-il d'être rentable ?
- Entraîner le modèle en laissant les séjours terminés par un décès. De combien l'AUC bouge-t-elle, et qu'est-ce que cela dit ?